In [1]:
import os
import json

def debug_paths():
    manifest_path = "./data/english_mini_batch/english_metadata.jsonl"
    vocals_dir = "./data/separated"

    print("🔍 --- STARTING PREPROCESS DIAGNOSTIC --- 🔍\n")

    # Check 1: Does the manifest exist?
    if not os.path.exists(manifest_path):
        print(f"❌ ERROR: Manifest not found at: {manifest_path}")
        return
    else:
        print(f"✅ FOUND Manifest file at: {manifest_path}")

    # Check 2: Does the vocals directory exist?
    if not os.path.exists(vocals_dir):
        print(f"❌ ERROR: Vocals directory not found at: {vocals_dir}")
        return
    else:
        print(f"✅ FOUND Vocals directory at: {vocals_dir}")
        print(f"   Files inside vocals dir (sample): {os.listdir(vocals_dir)[:3]}")

    # Check 3: Test a line from the manifest
    print("\n📖 Reading the first entry from your manifest...")
    with open(manifest_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if idx >= 1: 
                break
            try:
                entry = json.loads(line.strip())
                track_id = entry.get("track_id")
                
                # Let's look at what path the preprocessor is constructing:
                expected_vocal_file = os.path.join(vocals_dir, f"{track_id}_vocals.wav")
                
                print(f"👉 Manifest Track ID: {track_id}")
                print(f"👉 Preprocessor is looking for this physical file:\n   {expected_vocal_file}")
                
                if os.path.exists(expected_vocal_file):
                    print(f"✅ SUCCESS: The file physically exists! Loop shouldn't skip it.")
                else:
                    print(f"❌ ERROR: The file DOES NOT exist at that path.")
                    print("   Please check your file names inside your actual folder!")
            except Exception as e:
                print(f"❌ JSON Parsing Error: {str(e)}")

if __name__ == "__main__":
    debug_paths()

🔍 --- STARTING PREPROCESS DIAGNOSTIC --- 🔍

✅ FOUND Manifest file at: ./data/english_mini_batch/english_metadata.jsonl
✅ FOUND Vocals directory at: ./data/separated
   Files inside vocals dir (sample): ['53402892b5894ec89db54d79a2515702_vocals.wav', 'b615533ceecc4b6e9aaf5e8dcedaa5d0_vocals.wav', '6fbf8237cb4840a8a52499d28c40633d_vocals.wav']

📖 Reading the first entry from your manifest...
👉 Manifest Track ID: 68842abdf5ae47c4a9967aba6cfd7b3c
👉 Preprocessor is looking for this physical file:
   ./data/separated/68842abdf5ae47c4a9967aba6cfd7b3c_vocals.wav
✅ SUCCESS: The file physically exists! Loop shouldn't skip it.


In [ ]:
import os
import shutil
from datasets import load_dataset, Audio

def download_and_extract_dataset(audio_out_dir, lyrics_out_dir, language_filter="en"):
    """Downloads full-length audio tracks and complete ground-truth lyric text sheets
    directly from the HF benchmark repository and saves them as local physical files.
    """
    os.makedirs(audio_out_dir, exist_ok=True)
    os.makedirs(lyrics_out_dir, exist_ok=True)
    
    print("🤗 Fetching full-song metadata tracking blocks from Hugging Face...")
    
    # Load the test split of the master full-song dataset
    dataset = load_dataset("jamendolyrics/jam-alt", split="test")
    
    # Force Hugging Face to drop auto-decoding so we can capture the original file bytes natively
    dataset = dataset.cast_column("audio", Audio(decode=False))
    
    total_samples = len(dataset)
    download_count = 0
    
    print(f"📦 Found {total_samples} total tracks in the remote repository layout.")
    print("🚀 Commencing binary stream extraction...")
    
    for idx, sample in enumerate(dataset, start=1):
        song_name = sample.get("name") or sample.get("title") or f"track_{idx}"
        song_lang = sample.get("language")
        
        # Safe formatting for filename compatibility across operating systems
        safe_name = song_name.replace(" ", "_").replace("/", "-").strip() or f"track_{idx}"
        
        # Keep only the requested language. JAM-ALT uses short codes like "en".
        if language_filter and song_lang != language_filter:
            continue
            
        audio_meta = sample.get("audio")
        full_lyrics_text = sample.get("text")
        
        if not audio_meta or not full_lyrics_text:
            print(f"⚠️  [{idx}/{total_samples}] Skipping {safe_name}: Missing file components.")
            continue
            
        # Extract the original suffix format (.mp3 or .wav) directly from the HF path tracking dictionary
        original_ext = os.path.splitext(audio_meta["path"])[1]
        if not original_ext:
            original_ext = ".mp3" # Safe fallback default
            
        local_audio_path = os.path.join(audio_out_dir, f"{safe_name}{original_ext}")
        local_lyric_path = os.path.join(lyrics_out_dir, f"{safe_name}.txt")
        
        if os.path.exists(local_audio_path) and os.path.exists(local_lyric_path):
            print(f"⏭️  [{idx}/{total_samples}] Already exists, skipping: {safe_name}{original_ext}")
            continue
        
        # 1. Save audio. With Audio(decode=False), JAM-ALT usually gives bytes=None and a cached file path.
        try:
            audio_bytes = audio_meta.get("bytes")
            audio_path = audio_meta.get("path")
            if audio_bytes is not None:
                with open(local_audio_path, "wb") as audio_file:
                    audio_file.write(audio_bytes)
            elif audio_path and os.path.exists(audio_path):
                shutil.copyfile(audio_path, local_audio_path)
            else:
                raise FileNotFoundError(f"No audio bytes or cached file path for {safe_name}")
                
            # 2. Write the clean, unchopped reference lyric paragraph straight to disk
            with open(local_lyric_path, "w", encoding="utf-8") as text_file:
                text_file.write(full_lyrics_text.strip())
                
            download_count += 1
            print(f"✅ [{idx}/{total_samples}] Successfully Saved: {safe_name}{original_ext}")
            
        except Exception as e:
            print(f"❌ [{idx}/{total_samples}] Failed saving asset {safe_name}: {str(e)}")
            
    print("\n======================================================================")
    print("🏆 DOWNLOAD RUN COMPLETE")
    print("======================================================================")
    print(f"📥 Saved {download_count} validated asset pairs directly to disk.")
    print(f"📂 Audio Assets: {os.path.abspath(audio_out_dir)}")
    print(f"📂 Lyric Targets: {os.path.abspath(lyrics_out_dir)}\n")

# Run this cell to materialize only English JAM-ALT test tracks in an isolated folder.
download_and_extract_dataset(
    audio_out_dir="./data/jam_alt_english/raw_songs",
    lyrics_out_dir="./data/jam_alt_english/reference_lyrics",
    language_filter="en",
)

In [3]:
import os
import sys
from pathlib import Path
import torch
import torchaudio

# Direct runtime repo root injection boundary 
# REPO_ROOT = Path(__file__).resolve().parent
# if str(REPO_ROOT) not in sys.path:
#     sys.path.insert(0, str(REPO_ROOT))

from backend.utils.preprocess import chunk_audio
from backend.utils.demucs import separate_vocals

def export_and_verify_demucs_stems(input_track: str, output_dir: str = "./demucs_debug_output"):
    """
    Slices a target audio track, extracts vocals via Demucs, and exports 
    both the stereo and corrected mono waveforms straight to disk.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print("⏳ Slicing input track into raw RAM frames...")
    ram_chunks = chunk_audio(input_track, target_sr=16000)
    if not ram_chunks:
        print("❌ Failed to chunk audio track.")
        return
        
    print(f"🎸 Passing {len(ram_chunks)} frames into Demucs model processing line...")
    # This runs your standalone demucs function over the list arrays
    clean_vocal_chunks = separate_vocals(ram_chunks, sample_rate=16000)
    
    print(f"💾 Exporting clean chunks out to disk paths at {output_dir}...")
    for idx, vocal_array in enumerate(clean_vocal_chunks):
        
        # 1. Capture the raw structural dimensionality coming from the model
        print(f"👉 Chunk {idx:02d} | Source Array Shape: {vocal_array.shape}")
        
        # Save a Stereo configuration (Exactly what Demucs returned)
        stereo_tensor = torch.from_numpy(vocal_array)
        if stereo_tensor.ndim == 1:
            stereo_tensor = stereo_tensor.unsqueeze(0)
            
        stereo_path = os.path.join(output_dir, f"chunk_{idx:02d}_STEREO_raw.wav")
        torchaudio.save(stereo_path, stereo_tensor, 16000)
        
        # 2. Capture the exact math transformation we used to fix Whisper's distortion
        if vocal_array.ndim > 1 and vocal_array.shape[0] == 2:
            mono_array = vocal_array.mean(axis=0)  # Mix left + right down to 1D flat line
        else:
            mono_array = vocal_array.squeeze()
            
        mono_tensor = torch.from_numpy(mono_array).unsqueeze(0) # 1D to 2D single channel mapping [1, time]
        mono_path = os.path.join(output_dir, f"chunk_{idx:02d}_MONO_fixed.wav")
        torchaudio.save(mono_path, mono_tensor, 16000)
        
    print(f"\n🎉 Done! Stems exported successfully.")
    print(f"📁 Open this folder in Windows Explorer to play files: {os.path.abspath(output_dir)}")

if __name__ == "__main__":
    # 🎯 Update this path to any raw test mp3/wav song file sitting on your machine
    TARGET_SONG = "data/jam_alt_english/raw_songs/Avercage_-_Embers.mp3" 
    export_and_verify_demucs_stems(TARGET_SONG)

⏳ Slicing input track into raw RAM frames...


/home/vatsal/projects/asr/venv/lib/python3.10/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(


🎸 Passing 10 frames into Demucs model processing line...
💾 Exporting clean chunks out to disk paths at ./demucs_debug_output...
👉 Chunk 00 | Source Array Shape: (2, 1323000)
👉 Chunk 01 | Source Array Shape: (2, 1323000)
👉 Chunk 02 | Source Array Shape: (2, 1323000)
👉 Chunk 03 | Source Array Shape: (2, 1323000)
👉 Chunk 04 | Source Array Shape: (2, 1323000)
👉 Chunk 05 | Source Array Shape: (2, 1323000)
👉 Chunk 06 | Source Array Shape: (2, 1323000)
👉 Chunk 07 | Source Array Shape: (2, 1323000)
👉 Chunk 08 | Source Array Shape: (2, 1323000)
👉 Chunk 09 | Source Array Shape: (2, 1323000)

🎉 Done! Stems exported successfully.
📁 Open this folder in Windows Explorer to play files: /home/vatsal/projects/asr/demucs_debug_output
